# Notebook 3: Logistic Regression

## Overview
Logistic Regression is a **binary (or multi-class) classification** algorithm that models the probability of a class using the logistic (sigmoid) function.

### Key Concepts
* **Sigmoid function** σ(z) = 1 / (1 + e^(-z)) maps any real value to (0, 1).
* The decision boundary is where σ(z) = 0.5.
* Regularisation (C parameter in scikit-learn) prevents overfitting.
* Can be extended to **multi-class** via *one-vs-rest* or *softmax*.

## Dataset — Heart Disease (Kaggle)
**Source:** https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

Predicts whether a patient has heart disease (1) or not (0) based on clinical features.

### How to Download
```bash
pip install kaggle
# Place kaggle.json at ~/.kaggle/kaggle.json
kaggle datasets download -d johnsmith88/heart-disease-dataset \
  -p data/heart_disease --unzip
```

In [ ]:
# ─── Install (uncomment if needed) ─────────────────────────────────────────
# !pip install kaggle pandas scikit-learn matplotlib seaborn

# ─── Download dataset ───────────────────────────────────────────────────────
# !kaggle datasets download -d johnsmith88/heart-disease-dataset \
#   -p data/heart_disease --unzip

## 1. Imports and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────
DATA_PATH = 'data/heart_disease/heart.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Kaggle dataset loaded: {df.shape}")
except FileNotFoundError:
    # Fallback: use scikit-learn's built-in heart-disease-like dataset
    print("Kaggle file not found – using UCI Heart Disease via sklearn.")
    from sklearn.datasets import make_classification
    X_raw, y_raw = make_classification(
        n_samples=303, n_features=13, n_informative=8,
        n_redundant=2, random_state=42
    )
    cols = ['age','sex','cp','trestbps','chol','fbs',
            'restecg','thalach','exang','oldpeak','slope','ca','thal']
    df = pd.DataFrame(X_raw, columns=cols)
    df['target'] = y_raw
    print(f"Synthetic dataset created: {df.shape}")

df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Shape :", df.shape)
print("Missing values:\n", df.isnull().sum())
print("\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nClass balance: {df['target'].mean()*100:.1f}% positive cases")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','salmon'])
axes[0].set_title('Target Distribution (0=No Disease, 1=Disease)')
axes[0].set_xlabel('Target')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Correlation heatmap
corr = df.corr()
sns.heatmap(corr, annot=False, cmap='RdYlGn', ax=axes[1])
axes[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Handle any missing values
X.fillna(X.median(), inplace=True)

# Train / test split (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature scaling (important for logistic regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 4. Train Logistic Regression Model

In [ ]:
# C controls regularisation strength (larger C = less regularisation)
log_reg = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_train)

y_pred       = log_reg.predict(X_test_sc)
y_pred_proba = log_reg.predict_proba(X_test_sc)[:, 1]  # probability of positive class

print(f"Training Accuracy : {log_reg.score(X_train_sc, y_train):.4f}")
print(f"Test Accuracy     : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score     : {roc_auc_score(y_test, y_pred_proba):.4f}")

## 5. Evaluation Metrics

In [ ]:
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No Disease', 'Disease'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 6. Feature Coefficients (Importance)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(8, 5))
colours = ['salmon' if c < 0 else 'steelblue' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colours)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient')
plt.title('Logistic Regression Coefficients')
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 7. Effect of Regularisation (C parameter)

In [ ]:
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
train_acc, test_acc = [], []

for C in C_values:
    m = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    m.fit(X_train_sc, y_train)
    train_acc.append(m.score(X_train_sc, y_train))
    test_acc.append(m.score(X_test_sc, y_test))

plt.figure(figsize=(7, 4))
plt.semilogx(C_values, train_acc, marker='o', label='Train Accuracy')
plt.semilogx(C_values, test_acc,  marker='s', label='Test Accuracy')
plt.xlabel('C (inverse regularisation strength)')
plt.ylabel('Accuracy')
plt.title('Effect of Regularisation Strength on Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Cross-Validation

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_pipeline, X, y, cv=skf, scoring='accuracy')

print("5-Fold Cross-Validation Accuracies:", np.round(cv_scores, 4))
print(f"Mean Accuracy : {cv_scores.mean():.4f}")
print(f"Std Deviation : {cv_scores.std():.4f}")

## Summary

* Logistic Regression is a powerful baseline for **binary classification** tasks.
* Always scale features before training.
* Use **ROC-AUC** as the primary metric for imbalanced datasets.
* Tune the regularisation parameter `C` via cross-validation to avoid over/underfitting.